In [1]:
!pwd

/trinity/home/a.anokhin


In [1]:
import sys
import os
sys.path.append(os.path.abspath('.'))
sys.path.append(os.path.abspath('/trinity/home/a.anokhin/stage_2/pqn/multi-step-retrieval-rl-pqn-29_05/envs/babilong'))

In [2]:
from babilong_utils import TaskDataset
import numpy as np
from torch.utils.data import Dataset

import datasets
from os.path import join as join_path
from typing import List, Any, Tuple
from babilong_fix import QA2FixWrapper


class RetrSentenceSampler:
    def __init__(self,
                 dataset,
                 shuffle=True,
                 subsample_size = 100,
                 random_seed=42):
        self.sample_ind = 0
        self.dataset = dataset
        self.shuffle = shuffle
        self.subsample_size = subsample_size
        self.gen = np.random.default_rng(seed=random_seed)

    def get_sample(self, num_sentences):
        sample = []
        if num_sentences <= 0:
            return sample

        sentences = []
        while len(sentences) < num_sentences:
            n = min(num_sentences - len(sentences), self.subsample_size)
            new_sents = self.sentences_from_book(max_sentences_to_sample=n)
            sentences.extend(new_sents)

        return sentences[:num_sentences]

    def sentences_from_book(self, max_sentences_to_sample):
        sentences = []
        for attempt in range(100):
            book = self.next_book()
            if self.shuffle:
                if len(book) == 0:
                    continue
                i = self.gen.choice(len(book))
                book = book[i:i+max_sentences_to_sample]  # start from random position in text
            sentences.extend(book)
            if len(sentences) > 0:
                break
        else:
            raise ValueError(f'Tried to sample sentences from dataset {attempt} times but did not succeed')
        return sentences

    def next_book(self):
        if self.shuffle:
            sample_ind = self.gen.choice(len(self.dataset))
            sample = self.dataset[int(sample_ind)]['sentences']
        else:
            sample = self.dataset[int(self.sample_ind)]['sentences']
            self.sample_ind += 1
            self.sample_ind = self.sample_ind % len(self.dataset)
        return sample
    


def shuffle(noise: List[Any], facts :List[Any]) -> Tuple[List[Any], List[int]]:
    """
    Shuffles noise chunks with fact chunks while keeping relative order of facts intact
    """
    N_facts = len(facts)
    N = len(noise) + N_facts
    facts_idx = sorted(np.random.choice(N, size=N_facts, replace=False))
    all = []
    noise_i, fact_i = 0, 0
    for i in range(N):
        if fact_i < N_facts and i == facts_idx[fact_i]:
            all.append(facts[fact_i])
            fact_i += 1
        else:
            all.append(noise[noise_i])
            noise_i += 1
    return all, facts_idx



class RetrievalBabiLong(Dataset):

    @classmethod
    def create(cls, path, task, num_chunks, seed=42, noise_data_path="pg19/",
               split='train'):

        fact_dataset = TaskDataset(path, task, split)  # max_n_facts=10)
        if 'qa2' in task:
            # in Babi support facts may repeat several times.
            # the correct support fact among equal copies depends on the
            # relative order of facts.
            # QA2FixWrapper finds correct support facts for task QA2 and adds "references_idx" key to the sample
            fact_dataset = QA2FixWrapper(fact_dataset)
            #
        else:
            # It is too tedious to determine correct support fact for each task in BabiLong
            # the good heuristic is to select the last fact among the equal copies
            pass

        noise_path = join_path(path, noise_data_path, split)
        noise_dataset = datasets.load_from_disk(noise_path)
        noise_sampler = RetrSentenceSampler(noise_dataset, shuffle=True, random_seed=seed)

        dataset = cls(
            task_dataset=fact_dataset,
            noise_sentence_sampler=noise_sampler,
            num_sentences=num_chunks,
            random_seed=seed,
        )
        return dataset


    def __init__(
        self,
        task_dataset,
        noise_sentence_sampler,
        num_sentences,
        random_seed=42
    ):
        self.task_dataset = task_dataset
        self.noise_sampler = noise_sentence_sampler
        self.num_sentences = num_sentences
        if random_seed:
            self.gen = np.random.default_rng(seed=random_seed)

    def name(self):
        return 'babilong'

    def __getitem__(self, ind):
        sample = self.task_dataset[ind]
        sample_size = self.get_sample_size()
        num_facts = len(sample['facts'])
        num_noise = max(sample_size - num_facts, 0)
        noise_sentences = self.noise_sampler.get_sample(num_noise)
        sample['noise'] = noise_sentences

        chunks, facts_idx = shuffle(noise_sentences, sample['facts'])
        sample['chunks'] = chunks
        sample['facts_idx'] = facts_idx #all babi sentences

        if 'references_idx' in sample:
            new_ref_idx = [sample['facts_idx'][old_id] for old_id in sample['references_idx']]
            sample['references_idx'] = new_ref_idx # support facts from babi

        return sample

    def __len__(self):
        return len(self.task_dataset)

    def get_sample_size(self):
        if isinstance(self.num_sentences, list):
            return self.gen.choice(self.num_sentences)
        else:
            return self.num_sentences

    def get_partition_info(self):
        # Получаем из task_dataset
        task_ds = self.task_dataset
        if not hasattr(self.task_dataset,'get_partition_name'):
            task_ds = task_ds.task_dataset

        return task_ds.get_partition_name()

In [3]:
import torch
from transformers import AutoTokenizer

use_retrieval_dataset = True
num_tokens = 16000  #measure size of sample in tokens if use_retrieval_dataset == False
num_sentences = 10 #measure size of sample in sentences if use_retrieval_dataset == True
task = "qa2_two-supporting-facts"

train_path = f"/trinity/home/a.anokhin/babilong_test/recurrent-memory-transformer/data/tasks_1-20_v1-2/en-10k/qa2_two-supporting-facts_train.txt"
test_path = f"/trinity/home/a.anokhin/babilong_test/recurrent-memory-transformer/data/tasks_1-20_v1-2/en-10k/qa2_two-supporting-facts_test.txt"


# background text
lm_tokenizer = AutoTokenizer.from_pretrained('gpt2')

contriever_path = "/home/griver/projects/ml/nlp/contriever"
if contriever_path not in sys.path:
    sys.path.append(contriever_path)
#from src.contriever import Contriever
from transformers import AutoTokenizer

device = torch.device('cuda:0')
# embedder = Contriever.from_pretrained("facebook/contriever").to(device)
# embed_tokenizer = AutoTokenizer.from_pretrained("facebook/contriever")

#task_dataset_train = TaskDataset(train_path,) #max_n_facts=10)
task_dataset_test = TaskDataset(test_path,) #max_n_facts=10)


if use_retrieval_dataset:
    noise_dataset_name = "/trinity/home/a.anokhin/babilong_test/recurrent-memory-transformer/datasets/babilong/pg19-with-sentences/train"
    noise_dataset = datasets.load_from_disk(noise_dataset_name)
    noise_sampler_test = RetrSentenceSampler(noise_dataset)
    dataset_test = RetrievalBabiLong(task_dataset=task_dataset_test,
                                        noise_sentence_sampler=noise_sampler_test,
                                        num_sentences=num_sentences)
    print(f'retrieval dataset is loaded')
    sample = dataset_test[0]
    #print("first sample:")
    #print_dict(sample)
    #env = FaissRetrievalEnv(sample, embedder, embed_tokenizer, )

Loading dataset from disk:   0%|          | 0/47 [00:00<?, ?it/s]

retrieval dataset is loaded


In [4]:
dataset_test[0]

{'facts': array(['Mary got the milk there.', 'John moved to the bedroom.',
        'Sandra went back to the kitchen.',
        'Mary travelled to the hallway.'], dtype=object),
 'question': 'Where is the milk? ',
 'answer': 'hallway',
 'references': array(['Mary got the milk there.', 'Mary travelled to the hallway.'],
       dtype=object),
 'noise': ['A Baronitess junior sends word from the children\'s quarters that _Your\nFortune and Character_ is an amusing game, told by WILLIAM SHAKSPEARE,\nbut published by JOHN JAQUES & CO.--evidently not a descendant of the\n"melancholy JAQUES," for he would have "rail\'d on Lady Fortune in good\nterms" had the game been at his expense.',
  'Massa BLACKIE & SON send in a story by G. A. HENTY, always so\nHentytaining, entitled _When London Burned_.',
  'We all ken that when Rome\nburned NERO fiddled, but this hero--not an \'ero--had every opportunity\nof extinguishing--my Baronite means "distinguishing himself;" and our\ncavalier availed himself, a

In [92]:
sys.path.append(os.path.abspath('/trinity/home/a.anokhin/stage_2/pqn/multi-step-retrieval-rl-pqn-29_05'))
import numpy as np
from collections import namedtuple
from typing import Tuple, Dict, List, Any, Union
import torch.utils
from nltk.probability import gt_demo

# from rl.jax_text_env import TextEnv, TextMemory, TextMemoryItem
from rl.text_env import TextEnv, TextMemory, TextMemoryItem
from transformers import PreTrainedTokenizer, PreTrainedTokenizerFast


class GroundTruthReward:
    def __init__(self, only_at_max_step=False):
        super().__init__()
        self.only_at_max_step = only_at_max_step

    def reward(self, env, action):
        if self.only_at_max_step and (env.num_steps < env.max_steps):
            return 0.

        is_retrieved = []
        for r in env.references:
            is_retrieved.append(r in env.text_state)

        all_retrieved = all(is_retrieved)
        return float(all_retrieved)


class PositionalGTReward(GroundTruthReward):
    """
    This version takes into account position of the support facts.
    In babi tasks several events could have completely identical text descriptions,
    but only one of them can be considered a support fact/reference fact.

    I.E. Merry could visit the same location several times.
    But only the last event allows us to tell where she is at the end of the story.

    This reward takes into account temporal information that allows to distinguish
    true support facts, from similar events.
    """
    def reward(self, env, action):
        if self.only_at_max_step and (env.num_steps < env.max_steps):
            return 0.

        pred_sf = set(map(int, env.memory.item_ids))
        gt_sf = set(env.references_idx)
        return 1.0 if gt_sf.issubset(pred_sf) else 0.0


class BabilongEnv(TextEnv):

    def __init__(self,
                 dataset,
                 max_steps = 6,
                 max_chunks_count = 1000,
                 index_type = "random", # "absolute", "relative"
                 reward_model = GroundTruthReward()):
        
        super().__init__()

        self.dataset = dataset
        self.max_steps = max_steps
        self.max_chunks_count = max_chunks_count
        self.index_type = index_type
        # self.max_embed_length = max_embed_length
        # self.action_embed_length = action_embed_length
        self.reward_model = reward_model

        self.references = None
        self.question = None
        self.sentences = None
        self.facts_idx = None
       
        self.num_steps = 0

    def copy(self):
        return BabilongEnv(self.dataset, 
                           self.max_steps,
                           self.max_chunks_count,
                           self.index_type,
                           self.reward_model)

    def _init_from_sample(self, sample):
        self.references = list(sample['references'])
        self.question = sample['question']  # append as this is a single str
        self.answer = sample['answer']
        self.sentences = np.asarray(sample['chunks'])
        self.facts_idx = list(sample['facts_idx'])
        self.references_idx = sample.get('references_idx', None)
        # self.sentences.extend(sample['noise'])
        # self.sentences.extend(sample['facts'])
        # self.sentences.extend([
        #   f"Fact number {i}: "  + str(f) for i, f in enumerate(sample['facts'])
        # ])
        # self.ref_ids = []
        # for i, f in enumerate(sample['facts']):
        #     if f in self.references:
        #         self.ref_ids.append(i + len(sample['noise']))
        #
        # self.ref_ids = np.array(self.ref_ids)[len(self.ref_ids) - len(self.references):]


    def reset(self, new_sample=None) -> TextMemory:
        if new_sample is not None:
            self._init_from_sample(new_sample)

        elif self.dataset is not None:
            N = len(self.dataset)
            i = np.random.randint(N)
            new_sample = self.dataset[i]
            self._init_from_sample(new_sample)

        self.num_steps = 0

        self.refs_found = []
        self.text_state = []
        
        return super()._reset(self.question, self.sentences)
   

    def step(self, action: int):
        self.num_steps += 1

        done = self.num_steps >= self.max_steps
        
        text_memory, text_item, text_done = super()._step(action)
        self.text_state.append(self.sentences[action])

        r = self._reward(action)
        if r > 1e-5:
            done = True
    
        return text_memory, text_item, r, done or text_done

    @property
    def device(self):
        return self.embedder.device

    def _reward(self, action):
        return self.reward_model.reward(self, action)

    def get_sample_len(self, tokenizer):
        """
        Return total length of all texts in the current retrieval task
        """
        total_len = len(tokenizer(self.question)['input_ids'])
        total_len += sum(len(chunk) for chunk in tokenizer(list(self.sentences))['input_ids'])
        return total_len

In [93]:
env = BabilongEnv(dataset_test)

In [94]:
env

In [105]:
print("\nResetting environment...")
obs_memory = env.reset() # TextMemory object

print(f"Environment reset. Question: '{env.question}'")
print(f"Available sentences (chunks): {len(env.sentences)}")
# print("Sentences:", env.sentences)
print(f"Target reference indices: {env.references_idx}")
print(f"Target reference texts: {env.references}")

done = False
total_reward = 0
for step_num in range(env.max_steps):
    if done:
        break
    action = np.random.randint(0, len(env.sentences)) # Случайное действие
    print(f"\nStep {step_num + 1}, Agent takes action: {action} ('{env.sentences[action][:50]}...')")
    
    obs_memory, text_item, reward, done = env.step(action)
    
    total_reward += reward
    print(f"Received: text_item='{text_item[:50] if text_item else 'N/A'}...', reward={reward}, done={done}")
    if reward > 0:
        print(f"  Found a reference! Current refs_found: {env.refs_found}")

print(f"\nEpisode finished. Total reward: {total_reward}")


Resetting environment...
Environment reset. Question: 'Where is the apple? '
Available sentences (chunks): 14
Target reference indices: None
Target reference texts: ['Mary went back to the bedroom.', 'Mary left the apple.']

Step 1, Agent takes action: 12 ('Mary left the apple....')
Received: text_item='(933, None, None, 'Mary left the apple.')...', reward=0.0, done=False

Step 2, Agent takes action: 0 ('Mary moved to the kitchen....')
Received: text_item='(318, None, None, 'Mary moved to the kitchen.')...', reward=0.0, done=False

Step 3, Agent takes action: 4 ('Sandra moved to the bedroom....')
Received: text_item='(625, None, None, 'Sandra moved to the bedroom.')...', reward=0.0, done=False

Step 4, Agent takes action: 5 ('Mary went back to the bedroom....')
Received: text_item='(926, None, None, 'Mary went back to the bedroom.')...', reward=1.0, done=True
  Found a reference! Current refs_found: []

Episode finished. Total reward: 1.0
